In [1]:
from CaNH2_package import CaNH2_molecule
molecule = CaNH2_molecule(vibronic_states_to_include=("X","A"),N_range=(0,3))

In [2]:
molecule.solve_Hamiltonian(m_shift=True, Zeeman=False,B=(0,0,0.001))

In [3]:
states = molecule.eigenstates
Es = molecule.Es
TDMs = molecule.TDMs

for i, s in enumerate(states):
    s.E = Es[i]

In [4]:
X_states_111 = []
for i in range(len(states)):
    if states[i][0].elec == "X" and states[i].ka == 1 and states[i].kc == 1 and states[i][0].N == 1:
        X_states_111.append(states[i])
A_states_000 = []
for i in range(len(states)):
    if states[i][0].elec == "A" and states[i].ka == 0 and states[i].kc == 0 and states[i][0].N == 0:
        A_states_000.append(states[i])

In [5]:
energies = []
for s in X_states_111:
    energies.append(s.E * 1e6)
for s in A_states_000:
    energies.append(s.E * 1e6)

In [6]:
cycling_states = X_states_111 + A_states_000
i_ground0 = 1
i_ground1 = len(X_states_111)
i_exited0 = len(X_states_111) + 1
i_exicted1 = len(cycling_states)

In [7]:
import numpy as np

d = np.zeros((len(cycling_states),len(cycling_states),3),dtype=complex)

for i in range(len(cycling_states)):
    si = cycling_states[i]
    for j in range(len(cycling_states)):
        sj = cycling_states[j]
        d[i,j,0] = molecule.TDMs[-1][si, sj]
        d[i,j,1] = molecule.TDMs[0][si, sj]
        d[i,j,2] = molecule.TDMs[1][si, sj]

In [46]:
from scipy import constants as const

amu = const.value("atomic mass constant") #kg
h = const.h

Gamma = 17.5e6 #linewidth in Hz
m = 56 * amu
lam = 647e-9 # wavelength in m
k = 2 * np.pi / lam # wavevector


In [52]:
H_Zeemans = [None,None,None]
for id, d in enumerate(("x","y","z")):
    a = np.zeros([len(X_states_111),len(X_states_111)],dtype=complex)
    for i, si in enumerate(X_states_111):
        for j, sj in enumerate(X_states_111):
            a[i,j] = molecule.Zeeman_terms[d][si, sj]
    H_Zeemans[id] = a

In [53]:
a = np.array(H_Zeemans)
print(a.shape)

(3, 6, 6)


In [54]:
import numpy as np
import h5py


with h5py.File("CaNH2_data_export.h5", "w") as f:
    f.create_dataset("energies", data=energies)
    f.create_dataset("i_ground0", data=i_ground0)
    f.create_dataset("i_ground1", data=i_ground1)
    f.create_dataset("i_exited0", data=i_exited0)
    f.create_dataset("i_exicted1", data=i_exicted1)
    f.create_dataset("d", data=d)
    f.create_dataset("H_Zeemans", data=H_Zeemans)
    f.create_dataset("Gamma", data=Gamma)
    f.create_dataset("m", data=m)
    f.create_dataset("lam", data=lam)
    f.create_dataset("k", data=k)


